# 🧹 Data Cleaning Project - Employee Dataset
**Goal:** Clean a messy employee dataset by handling missing values, duplicates, invalid entries, and format issues.

### Problems in Raw Data:
- Missing values (age, salary, city, email)
- Duplicate rows
- Invalid emails
- Inconsistent gender values (M, male, Male, F, Female)
- Age outliers (e.g., age=999)
- Negative salary values
- Mixed date formats
- Extra whitespace in names

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime

# Load data
df = pd.read_csv('../data/raw_data.csv')
print(f'Shape: {df.shape}')
df.head(20)

In [ ]:
# Check missing values
print('Missing Values:')
print(df.isnull().sum())
print('\nData Types:')
print(df.dtypes)

In [ ]:
# Step 2: Remove duplicates
print(f'Before: {df.shape[0]} rows')
df = df.drop_duplicates()
print(f'After removing duplicates: {df.shape[0]} rows')

In [ ]:
# Step 3: Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Columns:', df.columns.tolist())

In [ ]:
# Step 4: Fix text columns
df['name'] = df['name'].str.strip().str.title()
df = df[~df['name'].isin(['N/A', 'Na', 'None', 'Unknown'])]
df = df.dropna(subset=['name'])

gender_map = {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}
df['gender'] = df['gender'].str.strip().str.lower().map(
    lambda x: gender_map.get(x, x.capitalize()) if isinstance(x, str) else x
)

df['department'] = df['department'].str.strip().str.title()
df['city'] = df['city'].str.strip().str.title()

print('Gender values:', df['gender'].unique())
print('Departments:', df['department'].unique())

In [ ]:
# Step 5: Validate emails
def is_valid_email(email):
    if pd.isna(email): return False
    return bool(re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', str(email).strip()))

df['email_valid'] = df['email'].apply(is_valid_email)
print('Invalid emails:', df[~df['email_valid']][['name', 'email']])
df.loc[~df['email_valid'], 'email'] = np.nan
df = df.drop(columns=['email_valid'])

In [ ]:
# Step 6: Fix age
df['age'] = pd.to_numeric(df['age'], errors='coerce')
print('Age outliers:', df[(df['age'] < 18) | (df['age'] > 65)][['name', 'age']])
df.loc[(df['age'] < 18) | (df['age'] > 65), 'age'] = np.nan
df['age'] = df['age'].fillna(df['age'].median()).astype(int)
print(f'Age range: {df["age"].min()} - {df["age"].max()}')

In [ ]:
# Step 7: Fix salary
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df.loc[df['salary'] < 0, 'salary'] = np.nan
df['salary'] = df.groupby('department')['salary'].transform(lambda x: x.fillna(x.median()))
df['salary'] = df['salary'].fillna(df['salary'].median()).astype(int)
print(f'Salary range: {df["salary"].min()} - {df["salary"].max()}')

In [ ]:
# Step 8: Fix dates
def parse_date(date_str):
    if pd.isna(date_str): return pd.NaT
    for fmt in ['%Y-%m-%d', '%d-%m-%Y', '%m/%d/%Y']:
        try: return datetime.strptime(str(date_str).strip(), fmt)
        except: continue
    return pd.NaT

df['joining_date'] = df['joining_date'].apply(parse_date)
df['joining_date'] = pd.to_datetime(df['joining_date'])
print('Date range:', df['joining_date'].min(), '-', df['joining_date'].max())

In [ ]:
# Step 9: Final fill
df['city'] = df['city'].fillna(df['city'].mode()[0])
df['email'] = df['email'].fillna('unknown@placeholder.com')

print('Final missing values:')
print(df.isnull().sum())

In [ ]:
# Step 10: Save cleaned data
df.to_csv('../outputs/cleaned_data.csv', index=False)
print(f'Saved! Final shape: {df.shape}')
df

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Employee Data - Cleaned Overview', fontsize=16, fontweight='bold')

dept_counts = df['department'].value_counts()
axes[0,0].bar(dept_counts.index, dept_counts.values, color='steelblue')
axes[0,0].set_title('Employees by Department')
axes[0,0].tick_params(axis='x', rotation=30)

gender_counts = df['gender'].value_counts()
axes[0,1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%')
axes[0,1].set_title('Gender Distribution')

axes[1,0].hist(df['salary'], bins=8, color='teal')
axes[1,0].set_title('Salary Distribution')

axes[1,1].hist(df['age'], bins=8, color='coral')
axes[1,1].set_title('Age Distribution')

plt.tight_layout()
plt.savefig('../outputs/visualization.png', dpi=150)
plt.show()